In [1]:
import torch
from diffusers import StableDiffusionPipeline
from torchmetrics.multimodal.clip_score import CLIPScore
from torchvision.transforms import ToTensor

# Initialize CLIPScore metric
metrica = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to("cuda")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


ModuleNotFoundError: No module named 'torchmetrics'

In [ ]:
# 1. Define os 6 prompts fixos (lembre de usar a palavra-gatilho no modelo LoRA)
prompts_base = [
    "a portrait of a man with a hat",
    "a castle on top of a mountain",
    "a woman playing guitar on a stage",
    "a dog running in a field at sunset",
    "a futuristic city with flying cars",
    "a serene forest lake with reflections",
]
# Adiciona a tag do estilo para as gerações com o LoRA
prompts_lora = [p + ", estilo_cordel style" for p in prompts_base]

# 2. Carrega o modelo base
pipe_base = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")

# 3. Gera as 6 imagens do modelo BASE usando seeds fixas (ex: 42)
imagens_base = []
for prompt in prompts_base:
    generator = torch.Generator("cuda").manual_seed(42) # MESMA SEED PARA TODOS
    img = pipe_base(prompt, generator=generator, num_inference_steps=30).images[0]
    imagens_base.append(img)

# 4. Libera memória e carrega o LoRA acoplado
# Delete pipe_base to free up memory (good practice)
del pipe_base
torch.cuda.empty_cache()

pipe_lora = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
pipe_lora.load_lora_weights("higosoares/atelie-estilo-lambelambe") # Seu modelo treinado

imagens_lora = []
#for prompt in prompts_lora:
#    generator = torch.Generator("cuda").manual_seed(42) # MESMA SEED DA BASE!
#    img = pipe_lora(prompt, generator=generator, num_inference_steps=30).images[0]
#    imagens_lora.append(img)

# Dica: Use a biblioteca PIL ou matplotlib para colar as imagens em uma grade 6x2 (lado a lado)

In [ ]:
# Calculate CLIP Scores for base images
clip_scores_base = []
print("Calculating CLIP Scores for base images:")
for i, img_pil in enumerate(imagens_base):
    # Convert PIL Image to Tensor (C, H, W) and then add batch dimension (1, C, H, W)
    img_tensor = ToTensor()(img_pil).unsqueeze(0).to("cuda") # Ensure it's on the correct device
    score = metrica(img_tensor, prompts_base[i])
    clip_scores_base.append(score.item())
    print(f"  Prompt: '{prompts_base[i]}' - CLIPScore: {score.item():.2f}")
print("\nAverage CLIPScore for base images:", sum(clip_scores_base) / len(clip_scores_base))

In [ ]:
# Calculate CLIP Scores for LoRA images
#clip_scores_lora = []
#print("\nCalculating CLIP Scores for LoRA images:")
#for i, img_pil in enumerate(imagens_lora):
#    img_tensor = ToTensor()(img_pil).unsqueeze(0).to("cuda")
#    score = metrica(img_tensor, prompts_lora[i])
#    clip_scores_lora.append(score.item())
#    print(f"  Prompt: '{prompts_lora[i]}' - CLIPScore: {score.item():.2f}")
#print("\nAverage CLIPScore for LoRA images:", sum(clip_scores_lora) / len(clip_scores_lora))